In [37]:
import os
import numpy as np
import pandas as pd

if os.path.basename(os.getcwd()) != "analiza_sprzedazy_w_sklepie":
    os.chdir("../..")

PLIK             = "dane/interim/fact_inka_final.parquet"
WYJSCIE_STAN     = "dane/interim/stan_magazynowy.parquet"
WYJSCIE_KONTROLA = "dane/interim/kontrola_remanent.parquet"
WYJSCIE_DZIENNY   = "dane/interim/stan_dzienny.parquet"

KOLUMNY_RUCHOW = ["SKU", "Data", "DokId", "Kolejnosc", "NrDok", "TypDok", "TypRuchu",
                  "IloscPlus", "IloscMinus", "CenaPoRab", "WplywNaStan", "MetodaLiczenia",
                  "AktywnyDok", "RuchMagazynowy"]
KOLUMNY_TOWARU = ["SKU", "NazwaTow", "NazwaAsort", "EAN", "AktywnyTow", "CzyWazony",
                  "CzyNiechciane", "CzyMartwyNaKoniecDanych", "KategoriaABCD"]
KLUCZ = ["SKU", "Data", "DokId", "Kolejnosc"]     # kolejność ruchów towaru w czasie
TOL = 0.001                                        # tolerancja porównania ilości

In [38]:
def wczytaj_ruchy(path=PLIK):
    """Ruchy wpływające na stan, z aktywnych dokumentów, bez pustych duplikatów BO, posortowane w czasie."""
    df = pd.read_parquet(path, columns=KOLUMNY_RUCHOW, filters=[("WplywNaStan", "==", 1)])
    df = df[df["AktywnyDok"] == 1]

    reset = df["MetodaLiczenia"] == "RESET_IP"
    pusty_bo = reset & (df["TypRuchu"] == "bo") & (df["IloscPlus"] == 0) & (df["IloscMinus"] == 0)
    print(f"Ruchów wpływających na stan (aktywne dokumenty): {len(df):,}; pominięto pustych BO: {int(pusty_bo.sum()):,}".replace(",", " "))
    df = df[~pusty_bo].sort_values(KLUCZ).reset_index(drop=True)

    assert df["RuchMagazynowy"].notna().all(), "RuchMagazynowy ma braki"
    r = df["MetodaLiczenia"] == "RESET_IP"
    assert np.allclose(df.loc[r, "RuchMagazynowy"], df.loc[r, "IloscPlus"]), "w resecie RuchMagazynowy powinno równać się IloscPlus"
    return df


def wylicz_stan_narastajaco(df):
    """Dodaje kolumny: delta, is_reset, epoka, stan (stan po tym ruchu).
    Epoka = odcinek między kolejnymi resetami (BO/remanent) tego towaru; w epoce stan = baza z resetu + suma delt."""
    df = df.copy()
    df["is_reset"] = df["MetodaLiczenia"] == "RESET_IP"
    df["delta"] = np.where(df["is_reset"], 0.0, df["RuchMagazynowy"])        # reset nie dodaje, tylko nadpisuje

    df["epoka"] = df.groupby("SKU")["is_reset"].cumsum()
    baza = pd.Series(np.where(df["is_reset"], df["IloscPlus"], np.nan))
    df["baza"] = baza.groupby(df["SKU"]).ffill().fillna(0.0).to_numpy()         # przed pierwszym resetem baza = 0
    df["stan"] = df["baza"] + df.groupby(["SKU", "epoka"])["delta"].cumsum()
    return df


def ostatnia_cena(df, maska, nazwa):
    """Cena z ostatniego wiersza spełniającego maskę, per SKU."""
    return df[maska].groupby("SKU")["CenaPoRab"].last().rename(nazwa)


def zbuduj_raport_stanow(df, path=PLIK):
    """Jeden wiersz na SKU: stan po ostatnim ruchu, data ostatniego ruchu i wycena (tylko stan dodatni)."""
    ostatni = df.groupby("SKU").tail(1).set_index("SKU")
    raport = pd.DataFrame({"StanMagazynowy": ostatni["stan"], "DataOstatniegoRuchu": ostatni["Data"]})
    raport["OstatniaCenaSprzedazy"] = ostatnia_cena(df, df["TypDok"] == 21, "OstatniaCenaSprzedazy")
    raport["OstatniaCenaZakupu"] = ostatnia_cena(df, df["TypDok"] == 2, "OstatniaCenaZakupu")

    # kiedy stan był ostatnio „odnowiony” spisem (BO lub remanent); od tego czasu stan to suma ruchów i dryfuje
    raport["DataOstatniegoResetu"] = df[df["is_reset"]].groupby("SKU")["Data"].last()
    raport["DniOdResetu"] = (df["Data"].max() - raport["DataOstatniegoResetu"]).dt.days

    dodatni = raport["StanMagazynowy"].clip(lower=0)
    raport["CzyStanUjemny"] = (raport["StanMagazynowy"] < -TOL).astype("int8")
    raport["WartoscSprzedazowa"] = dodatni * raport["OstatniaCenaSprzedazy"]
    raport["WartoscZakupu"] = dodatni * raport["OstatniaCenaZakupu"]

    towar = pd.read_parquet(path, columns=KOLUMNY_TOWARU).drop_duplicates("SKU").set_index("SKU")
    raport = towar.join(raport, how="right").reset_index()
    return raport.sort_values("WartoscZakupu", ascending=False, na_position="last").reset_index(drop=True)


def sprawdz_zgodnosc_z_resetami(df):
    """Kontrola jakości: dla każdego resetu (BO/remanent) porównuje stan WYLICZONY tuż przed nim ze stanem
    SPISANYM w systemie (IloscMinus w wierszu resetu). diff = 0 oznacza zgodność."""
    df = df.copy()
    df["stan_przed"] = df.groupby("SKU")["stan"].shift(1).fillna(0.0)
    r = df[df["is_reset"]].copy()
    r["diff"] = r["stan_przed"] - r["IloscMinus"]
    return r[["SKU", "NazwaTow" if "NazwaTow" in r else "SKU", "Data", "NrDok", "TypRuchu", "stan_przed", "IloscMinus", "diff", "IloscPlus"]].loc[:, lambda t: ~t.columns.duplicated()]

In [39]:
ruchy = wczytaj_ruchy()
ruchy = wylicz_stan_narastajaco(ruchy)

raport = zbuduj_raport_stanow(ruchy)
kontrola = sprawdz_zgodnosc_z_resetami(ruchy)

os.makedirs("dane/interim", exist_ok=True)
raport.to_parquet(WYJSCIE_STAN, index=False)
kontrola.to_parquet(WYJSCIE_KONTROLA, index=False)

print(f"Towarów w raporcie: {len(raport):,}".replace(",", " "))

Ruchów wpływających na stan (aktywne dokumenty): 3 735 781; pominięto pustych BO: 16 912
Towarów w raporcie: 12 481


## Kontrola: stan wyliczony a spisany w systemie
Przy każdym BO i remanencie system zapisuje, ile towaru *miał* przed spisem (`IloscMinus`). Jeśli nasz stan tuż przed resetem się zgadza, ruchy są policzone poprawnie. **BO zgadza się zawsze z założenia**: BO to początek danych (stan przed nim wynosi 0, `IloscMinus` też). Prawdziwym testem jest remanent.

In [40]:
print("Daty BO w danych:", sorted(kontrola.loc[kontrola["TypRuchu"] == "bo", "Data"].dt.date.astype(str).unique()))
for typ in ("bo", "remanent"):
    c = kontrola[kontrola["TypRuchu"] == typ]
    ok = (c["diff"].abs() <= TOL)
    print(f"{typ}: {int(ok.sum()):,}/{len(c):,} zgodnych ({ok.mean()*100:.1f} %)".replace(",", " "))

rem = kontrola[kontrola["TypRuchu"] == "remanent"].copy()
rem["Zgodny"] = rem["diff"].abs() <= TOL
roczna = rem.groupby(rem["Data"].dt.year)["Zgodny"].agg(remanentow="size", zgodnych="sum")
roczna["zgodnych_%"] = (roczna["zgodnych"] / roczna["remanentow"] * 100).round(1)
print("\nRemanenty wg roku:"); print(roczna.to_string())

nz = rem[~rem["Zgodny"]]
print(f"\nNiezgodnych remanentów: {len(nz)} na {nz['SKU'].nunique()} towarach; błąd mediana {nz['diff'].median():.1f} szt., "
      f"|błąd| <= 2 szt. w {(nz['diff'].abs() <= 2).mean()*100:.0f} % przypadków, największy {nz['diff'].abs().max():.0f} szt.")

Daty BO w danych: ['2023-01-01']
bo: 5 418/5 418 zgodnych (100.0 %)
remanent: 26 653/27 709 zgodnych (96.2 %)

Remanenty wg roku:
      remanentow  zgodnych  zgodnych_%
Data                                  
2023        7220      7009        97.1
2024       19669     18997        96.6
2025         655       488        74.5
2026         165       159        96.4

Niezgodnych remanentów: 1056 na 506 towarach; błąd mediana -1.0 szt., |błąd| <= 2 szt. w 63 % przypadków, największy 353 szt.


In [41]:
rem = kontrola[(kontrola["TypRuchu"] == "remanent") & ~((kontrola["IloscPlus"] == 0) & (kontrola["IloscMinus"] == 0))].copy()
rem["korekta"] = rem["IloscPlus"] - rem["IloscMinus"]          # spis fizyczny − system
print("spis = system:", (rem["korekta"].abs() <= TOL).mean())
print("odtworzenie systemu:", (rem["diff"].abs() <= TOL).mean())

spis = system: 0.5357466545822893
odtworzenie systemu: 0.9494055552593699


- Odtworzenie systemu: 94,9%. To sprawdza nasz kod. W 94,9% remanentów stan, który wyliczyliśmy z ruchów, jest identyczny z tym, który miał system Inki tuż przed spisem.
- Spis = system: 53,6%. To sprawdza stan magazynowy w systemie, a nie nasz kod. W 53,6% spisów to, co policzono na półce, było równe temu, co miał system. W pozostałych 46,4% ktoś musiał stan skorygować. Jak już wcześniej policzyłem: połowa tych korekt to 1 sztuka, ale korekty są głównie ujemne (towaru brakowało częściej, niż go przybywało).

## Raport stanów
Stan ujemny oznacza, że w danych brakuje przyjęcia albo ruch został zapisany poza kolejnością (np. sprzedaż przed zarejestrowaniem dostawy). Taki stan zostaje w `StanMagazynowy` bez zmian (to sygnał jakości), ale do wyceny liczymy go jako 0.

In [42]:
ujemne = raport[raport["CzyStanUjemny"] == 1]
dodatnie = raport[raport["StanMagazynowy"] > TOL]
print(f"Towarów: {len(raport):,} | stan dodatni: {len(dodatnie):,} | stan zero: {len(raport) - len(dodatnie) - len(ujemne):,} | stan ujemny: {len(ujemne):,} (razem {ujemne['StanMagazynowy'].sum():,.0f} szt.)".replace(",", " "))
print(f"Stan dodatni razem: {dodatnie['StanMagazynowy'].sum():,.0f} szt.".replace(",", " "))
print(f"Wartość magazynu w cenie zakupu (ostatnie PZ):  {raport['WartoscZakupu'].sum():>12,.0f} zł".replace(",", " "))
print(f"Wartość magazynu w cenie sprzedaży (ostatnia):   {raport['WartoscSprzedazowa'].sum():>12,.0f} zł".replace(",", " "))
brak_pz = dodatnie[dodatnie["OstatniaCenaZakupu"].isna()]
print(f"Towary ze stanem dodatnim bez ceny zakupu (nigdy nie było PZ): {len(brak_pz)}, ich wartość sprzedażowa {brak_pz['WartoscSprzedazowa'].sum():,.0f} zł".replace(",", " "))

print("\nNajwiększe pozycje wg wartości w cenie zakupu:")
print(raport.head(10)[["SKU", "NazwaTow", "StanMagazynowy", "OstatniaCenaZakupu", "WartoscZakupu", "CzyMartwyNaKoniecDanych"]].round(2).to_string())

print("\nJak świeży jest stan: dni od ostatniego BO/remanentu (towary ze stanem dodatnim):")
korb = pd.cut(dodatnie["DniOdResetu"], [-1, 90, 180, 365, 10_000], labels=["do 90 dni", "91-180", "181-365", "ponad 365"]).astype(object)
korb = korb.where(dodatnie["DniOdResetu"].notna(), "brak BO/remanentu")
sw = dodatnie.assign(Swiezosc=korb).groupby("Swiezosc").agg(Towary=("SKU", "size"), Sztuki=("StanMagazynowy", "sum"), WartoscZakupu=("WartoscZakupu", "sum"))
sw["Wartosc_%"] = sw["WartoscZakupu"] / sw["WartoscZakupu"].sum() * 100
print(sw.round(1).to_string())

print("\nWartość zakupu wg klasy ABCD i martwych towarów (stan dodatni):")
print(raport.groupby("KategoriaABCD")[["WartoscZakupu", "WartoscSprzedazowa"]].sum().round(0).to_string())
print("\nMartwe na koniec danych (CzyMartwyNaKoniecDanych = 1):",
      f"{int(dodatnie['CzyMartwyNaKoniecDanych'].sum())} towarów ze stanem dodatnim, wartość zakupu {dodatnie.loc[dodatnie['CzyMartwyNaKoniecDanych'] == 1, 'WartoscZakupu'].sum():,.0f} zł".replace(",", " "))

Towarów: 12 481 | stan dodatni: 5 745 | stan zero: 6 549 | stan ujemny: 187 (razem -2 410 szt.)
Stan dodatni razem: 74 277 szt.
Wartość magazynu w cenie zakupu (ostatnie PZ):       304 976 zł
Wartość magazynu w cenie sprzedaży (ostatnia):        394 374 zł
Towary ze stanem dodatnim bez ceny zakupu (nigdy nie było PZ): 37  ich wartość sprzedażowa 1 163 zł

Największe pozycje wg wartości w cenie zakupu:
     SKU                                  NazwaTow  StanMagazynowy  OstatniaCenaZakupu  WartoscZakupu  CzyMartwyNaKoniecDanych
0   4685       WÓDKA ŻUBRÓWKA BIAŁA 40% 500ML CEDC            85.0               20.99        1784.15                        0
1     65            Chleb mieszany 0.6 kg PRECELEK           245.5                3.90         957.45                        0
2     71                   Chleb RAZOWY MIX 0,4kg.           172.0                5.35         920.20                        0
3   2437          PIWO PUSZ HARNAŚ 500ML CARLSBERG           366.0                2.50 

## Stan dzienny (`SKU × Data`)
Dla każdego towaru i każdego dnia kalendarza z `fact_inka_final` (także dni bez ruchu):

| kolumna | znaczenie |
|---|---|
| `StanPoczatekDnia` | stan przed pierwszym ruchem dnia (= stan na koniec poprzedniego dnia); **tej kolumny używamy jako cechy modelu**, bo nie zawiera sprzedaży z danego dnia |
| `RuchWCiaguDnia` | suma ruchów dnia ze znakiem (przyjęcia +, sprzedaż, RW i straty −, korekty i przesunięcia); bez remanentów |
| `KorektaRemanentu` | o ile BO lub remanent zmienił stan wyliczony z ruchów (inny niż 0 tylko w dniach z resetem) |
| `StanKoniecDnia` | stan po wszystkich ruchach dnia; zawsze `StanPoczatekDnia + RuchWCiaguDnia + KorektaRemanentu` |
| `CzyStanUjemny` | 1, jeśli stan na koniec dnia < 0 |
| `DniOdResetu` | dni od ostatniego BO/remanentu tego towaru (na koniec dnia); im więcej, tym mniej wiarygodny stan |

Przed pierwszym ruchem towaru stan wynosi 0. Dni, w których są ruchy, a których nie ma w kalendarzu pliku (np. przyjęcie po ostatniej sprzedaży towaru), też trafiają do tabeli, żeby ostatni dzień zgadzał się z raportem stanów.

**Uwaga o wycieku.** `StanKoniecDnia` i `RuchWCiaguDnia` zawierają sprzedaż tego samego dnia, więc nie wolno ich podawać modelowi prognozy sprzedaży tego dnia. Służą do opisu i kontroli.

In [43]:
def zbuduj_stan_dzienny(df, path=PLIK):
    """Tabela SKU x Data: stan na początek dnia, ruch w ciągu dnia, korekta remanentu, stan na koniec dnia.
    df = wynik wylicz_stan_narastajaco() (ruchy posortowane po SKU, Data, DokId, Kolejnosc)."""
    dzien = df.groupby(["SKU", "Data"]).agg(
        StanKoniecDnia=("stan", "last"),                 # stan po ostatnim ruchu dnia
        RuchWCiaguDnia=("delta", "sum"),                 # delta = 0 w wierszach resetu, więc remanenty tu nie wchodzą
        Reset=("is_reset", "max"),
    ).reset_index()
    dzien["DataResetu"] = dzien["Data"].where(dzien["Reset"])

    kal = pd.read_parquet(path, columns=["SKU", "Data"]).drop_duplicates()
    kal = pd.concat([kal, dzien[["SKU", "Data"]]]).drop_duplicates().sort_values(["SKU", "Data"]).reset_index(drop=True)
    sd = kal.merge(dzien[["SKU", "Data", "StanKoniecDnia", "RuchWCiaguDnia", "DataResetu", "Reset"]], on=["SKU", "Data"], how="left")
    sd["DzienResetu"] = sd["Reset"].fillna(False).astype(bool)
    sd["DzienResetu"] = sd["DzienResetu"].astype('int8')
    assert len(sd) == len(kal) and not sd.duplicated(["SKU", "Data"]).any()

    sd["StanKoniecDnia"] = sd.groupby("SKU")["StanKoniecDnia"].ffill().fillna(0.0)   # dzień bez ruchu = stan z poprzedniego dnia
    sd["RuchWCiaguDnia"] = sd["RuchWCiaguDnia"].fillna(0.0)
    # dni bez wiersza w tabeli nie mają ruchów (każdy dzień z ruchem jest w tabeli), więc poprzedni wiersz to koniec poprzedniego dnia
    sd["StanPoczatekDnia"] = sd.groupby("SKU")["StanKoniecDnia"].shift(1).fillna(0.0)
    sd["KorektaRemanentu"] = sd["StanKoniecDnia"] - sd["StanPoczatekDnia"] - sd["RuchWCiaguDnia"]
    sd.loc[sd["KorektaRemanentu"].abs() < 1e-6, "KorektaRemanentu"] = 0.0             # szum zmiennoprzecinkowy

    sd["DataResetu"] = sd.groupby("SKU")["DataResetu"].ffill()
    sd["DniOdResetu"] = (sd["Data"] - sd["DataResetu"]).dt.days.astype("Int16")
    sd["CzyStanUjemny"] = (sd["StanKoniecDnia"] < -TOL).astype("int8")
    for k in ("StanPoczatekDnia", "RuchWCiaguDnia", "KorektaRemanentu", "StanKoniecDnia"):
        sd[k] = sd[k].astype("float32")
    return sd[["SKU", "Data", "StanPoczatekDnia", "RuchWCiaguDnia", "KorektaRemanentu", "StanKoniecDnia", "CzyStanUjemny", "DniOdResetu", "DzienResetu"]]


stan_dzienny = zbuduj_stan_dzienny(ruchy)

# Kontrola 1: korekta jest różna od 0 tylko w dniach z BO/remanentem
zle = stan_dzienny[(stan_dzienny["KorektaRemanentu"].abs() > 0.01) & ~stan_dzienny["DzienResetu"]]
assert len(zle) == 0, f"korekta poza dniem resetu: {len(zle)} wierszy"
# Kontrola 2: tożsamość początek + ruch + korekta = koniec (z tolerancją float32)
tozsamosc = (stan_dzienny["StanPoczatekDnia"] + stan_dzienny["RuchWCiaguDnia"] + stan_dzienny["KorektaRemanentu"] - stan_dzienny["StanKoniecDnia"]).abs()
assert tozsamosc.max() < 0.05, f"tożsamość nie zachodzi, max różnica {tozsamosc.max():.3f}"
# Kontrola 3: ostatni dzień każdego towaru = stan z raportu
ost = stan_dzienny.groupby("SKU").tail(1).set_index("SKU")["StanKoniecDnia"]
roznica = (ost - raport.set_index("SKU")["StanMagazynowy"].reindex(ost.index)).abs()
assert roznica.max() < 0.01, f"ostatni dzień nie zgadza się z raportem (max różnica {roznica.max():.3f})"
# Kontrola 4: początek dnia = koniec poprzedniego dnia tego towaru
poprz = stan_dzienny.groupby("SKU")["StanKoniecDnia"].shift(1).fillna(0.0)
assert (poprz - stan_dzienny["StanPoczatekDnia"]).abs().max() < 1e-4

print(f"Wierszy: {len(stan_dzienny):,} | towarów: {stan_dzienny['SKU'].nunique():,} | zakres: {stan_dzienny['Data'].min().date()} … {stan_dzienny['Data'].max().date()}".replace(",", " "))
print(f"Dni z resetem (BO/remanent): {int(stan_dzienny['DzienResetu'].sum()):,}; z niezerową korektą: {int((stan_dzienny['KorektaRemanentu'].abs() > 0.01).sum()):,}".replace(",", " "))
print(f"Dni ze stanem ujemnym: {stan_dzienny['CzyStanUjemny'].mean()*100:.1f} % wierszy; towarów, które kiedykolwiek miały stan ujemny: {stan_dzienny.loc[stan_dzienny['CzyStanUjemny'] == 1, 'SKU'].nunique():,}".replace(",", " "))
print(f"Dni, w których stan jest oparty na spisie sprzed ponad 365 dni: {(stan_dzienny['DniOdResetu'] > 365).mean()*100:.1f} % wierszy")
print(stan_dzienny[["StanPoczatekDnia", "RuchWCiaguDnia", "KorektaRemanentu", "StanKoniecDnia"]].describe().round(2).to_string())

stan_dzienny.drop(columns="DzienResetu").to_parquet(WYJSCIE_DZIENNY, compression="zstd", index=False)
print("Zapisano:", WYJSCIE_DZIENNY)

Wierszy: 7 091 162 | towarów: 12 481 | zakres: 2023-01-01 … 2026-01-31
Dni z resetem (BO/remanent): 33 112; z niezerową korektą: 14 364
Dni ze stanem ujemnym: 1.6 % wierszy; towarów  które kiedykolwiek miały stan ujemny: 2 851
Dni, w których stan jest oparty na spisie sprzed ponad 365 dni: 6.9 % wierszy
       StanPoczatekDnia  RuchWCiaguDnia  KorektaRemanentu  StanKoniecDnia
count        7091162.00      7091162.00        7091162.00      7091162.00
mean              11.71            0.00              0.01           11.72
std               21.71            4.76              1.40           21.72
min             -849.00         -307.00          -2044.60         -890.00
25%                3.00            0.00              0.00            3.00
50%                8.00            0.00              0.00            8.00
75%               14.00            0.00              0.00           14.00
max             2044.60         1440.00           1955.60         2044.60
Zapisano: dane/interim/stan_d

# Podgląd ruchów dziennych oraz z resztą danych z fact_inka_final"

In [44]:

df = pd.read_parquet("dane/interim/fact_inka_final.parquet")
df = df.merge(stan_dzienny, on=["SKU", "Data"], how="left")
df.to_parquet("dane/interim/fact_inka_final_stan_dzienny.parquet", index=False)

In [45]:
df.sample(5)

,SKU,Data,DokId,Kolejnosc,NrPozycji,TypPoz,IloscPlus,IloscMinus,CenaPoRab,Wartosc,...,CzyPromocja,TypPromocji,ObnizkaPct,StanPoczatekDnia,RuchWCiaguDnia,KorektaRemanentu,StanKoniecDnia,CzyStanUjemny,DniOdResetu,DzienResetu
9069176,79184,2024-11-03,-1,0,0,4,0.0,0.0,4.7524,0.0,...,0,brak,0.0,18.0,0.0,0.0,18.0,0,<NA>,0
4776328,37088,2024-03-11,-1,0,0,4,0.0,0.0,27.6341,0.0,...,0,brak,0.0,2.0,0.0,0.0,2.0,0,15,0
1722589,6645,2024-06-24,1713793,3,3,4,1.0,0.0,3.8000,3.8,...,0,brak,0.0,10.0,-2.0,0.0,8.0,0,120,0
1312420,5567,2023-12-11,-1,0,0,4,0.0,0.0,1.4900,0.0,...,0,brak,0.0,10.0,0.0,0.0,10.0,0,295,0
7066708,54293,2024-03-30,-1,0,0,4,0.0,0.0,4.8699,0.0,...,0,brak,0.0,0.0,0.0,0.0,0.0,0,34,0
